# RT-DETR 베이스라인

## 목적

- transformer 계열 검출 모델인 RT-DETR로 베이스라인을 만든다. 팀에서 돌리는 YOLO26(1단계 CNN), Faster R-CNN(2단계 CNN)과 계열이 다른 모델이다.
- 데이터 변환부터 학습, test 제출, val 분석까지 한 번 끝까지 이어 본다.

## 데이터

- 대회 train 데이터에 팀 공용 라벨 수정 목록(`data/label_fix/label_fixes.csv`)을 적용해서 쓴다.
- train/val 분할은 1차 EDA에서 고정한 분할(seed 410, `data/processed/split.csv`)을 그대로 쓴다. train 185장, val 47장이다.

## 목차

| 섹션 | 내용 |
|---|---|
| 0. 사전 설정 | 라이브러리, 경로, GPU, 한글 글꼴 |
| 1. RT-DETR를 고른 이유 | 팀 안에서의 위치, 다른 후보와 비교, 한계 |
| 2. 데이터 준비 | 라벨 읽기, 라벨 수정 적용, 분할 붙이기, YOLO 형식 변환, 변환 확인 |
| 3. 학습 설정 | 학습 인자, 확인 실행, W&B 기록 |
| 4. 학습 | GPU 메모리 확인, seed별 학습, 학습 곡선 |
| 5. test 추론과 제출 | seed별 test 예측과 제출 CSV, 파일 점검, Kaggle 제출과 점수 |
| 6. val 분석 | 클래스별 AP(mAP75-95 포함), 박스 단위 맞음/틀림 |
| 7. 예측 보기 | val 사진에 예측과 정답 그리기 |
| 8. 정리 | 결과와 다음에 할 것 |

## 실행 방법

- 대회 데이터 압축을 `data/raw` 아래에 풀어 둔다.
- `data/processed/split.csv`와 `data/label_fix/label_fixes.csv`가 있어야 한다.
- W&B에 기록하려면 터미널에서 `wandb login`을 한 번 해 둔다. W&B 팀 `codeit-prj1-team2`의 멤버여야 한다.
- 처음부터 순서대로 실행한다. 3-1의 `SMOKE_TEST = True`로 짧은 확인 실행을 먼저 하고, 문제가 없으면 `False`로 바꿔 본 학습을 한다.
- 학습이 끝난 뒤에는 4번을 다시 실행하지 않는다(학습을 다시 시작한다). 5번부터는 0번 두 셀과 3-1, 3-2만 먼저 실행하면 된다.

## 0. 사전 설정

- 경로는 `pyproject.toml`이 있는 레포 최상위 폴더를 기준으로 잡는다. 노트북을 어느 폴더에서 열어도 같은 경로를 쓰기 위해서다. 출력할 때는 내 컴퓨터의 절대경로 대신 레포 기준 상대경로로 보여 준다.
- 학습에 필요한 파일(원본 데이터, 분할 파일, 라벨 수정 목록)이 있는지 먼저 확인한다. 없으면 학습 도중이 아니라 여기서 멈추게 한다.
- 실행 환경(Python, PyTorch, CUDA, Ultralytics, W&B, GPU)을 출력해 둔다. 결과가 다르게 나올 때 환경 차이부터 확인하기 위해서다.
- 사전학습 가중치는 레포의 `weights` 폴더에 둔다. 없으면 처음 학습할 때 내려받는다.
- 그래프의 한글이 깨지지 않게 한글 글꼴을 설정한다.

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import wandb
import yaml
from matplotlib import font_manager
from matplotlib.patches import Rectangle
from PIL import Image
from ultralytics import RTDETR, settings

# 경로: pyproject.toml이 있는 폴더를 레포 최상위로 본다
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
assert ROOT is not None, "pyproject.toml이 있는 레포 폴더를 찾지 못했습니다. 레포 안에서 노트북을 실행하세요."


def rel(path):
    """레포 최상위 기준 상대경로를 / 구분자로 돌려준다. (내 컴퓨터의 절대경로를 출력하지 않기 위해)"""
    return Path(path).relative_to(ROOT).as_posix()


found = [p for p in sorted((ROOT / "data" / "raw").rglob("train_images")) if p.is_dir()]
assert len(found) == 1, f"data/raw 아래에 train_images 폴더가 1개여야 합니다. 찾은 수: {len(found)}"
RAW_DIR = found[0].parent
TRAIN_IMG_DIR = RAW_DIR / "train_images"
TRAIN_ANN_DIR = RAW_DIR / "train_annotations"
TEST_IMG_DIR = RAW_DIR / "test_images"

SPLIT_CSV = ROOT / "data" / "processed" / "split.csv"                          # 1차 EDA 분할 (seed 410)
FIX_CSV = ROOT / "data" / "label_fix" / "label_fixes.csv"                      # 팀 공용 라벨 수정 목록
YOLO_DIR = ROOT / "data" / "processed" / "yolo"                                # YOLO 형식 데이터를 만들 곳
DATA_YAML = YOLO_DIR / "data.yaml"                                             # YOLO 데이터 설정 파일 (2-4에서 만든다)
RUNS_DIR = ROOT / "runs"                                                       # 학습 결과를 쌓을 곳
WEIGHTS_DIR = ROOT / "weights"                                                 # 사전학습 가중치를 둘 곳

for path in [TRAIN_IMG_DIR, TRAIN_ANN_DIR, TEST_IMG_DIR, SPLIT_CSV, FIX_CSV]:
    assert path.exists(), f"없습니다: {rel(path)}"

DEVICE = 0 if torch.cuda.is_available() else "cpu"  # 0번 GPU. GPU가 없으면 CPU

print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__, "/ CUDA:", torch.version.cuda)
print("Ultralytics:", ultralytics.__version__)
print("W&B        :", wandb.__version__)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU        :", gpu.name, f"/ 메모리 {gpu.total_memory / 1024**3:.1f} GiB")
else:
    print("GPU        : 없음 (CPU로 학습하면 매우 느립니다)")
print()
for name, path in [("원본 데이터", RAW_DIR), ("분할 파일", SPLIT_CSV), ("라벨 수정 목록", FIX_CSV),
                   ("YOLO 데이터", YOLO_DIR), ("학습 결과", RUNS_DIR), ("사전학습 가중치", WEIGHTS_DIR)]:
    print(f"{name:10}: {rel(path)}")

In [ ]:
KOREAN_FONTS = ["Malgun Gothic", "AppleGothic", "NanumGothic"]
installed = {f.name for f in font_manager.fontManager.ttflist}
FONT = next((name for name in KOREAN_FONTS if name in installed), None)
assert FONT is not None, f"한글 글꼴을 찾지 못했습니다. {KOREAN_FONTS} 중 하나를 설치한 뒤 커널을 다시 시작하세요."

plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = False  # 한글 글꼴에서 음수 기호가 깨지지 않도록

print("운영체제:", platform.system(), "/ 글꼴:", FONT)

## 1. RT-DETR를 고른 이유

#### 팀 안에서의 위치

| 모델 | 계열 | 예측 방식 |
|---|---|---|
| YOLO26 | 1단계 CNN | 이미지를 한 번 보고 위치와 클래스를 함께 예측한다. |
| Faster R-CNN | 2단계 CNN | 알약이 있을 만한 영역을 먼저 고르고(1단계), 그 영역의 클래스와 bbox를 다듬는다(2단계). |
| RT-DETR | transformer | CNN으로 이미지 특징을 뽑은 뒤 transformer가 정해진 개수의 예측(쿼리)을 한 번에 낸다. 학습할 때 예측과 정답 bbox를 일대일로 짝지어(헝가리안 매칭) 오차를 계산한다. |

- RT-DETR는 미리 정한 기준 박스(앵커)와, 같은 물체에 겹친 예측을 지우는 후처리(NMS)를 쓰지 않는다. 앞의 두 모델과 예측 방식이 달라서, 세 모델을 비교하면 계열에 따른 차이를 볼 수 있다.

#### DETR 계열 중 RT-DETR를 고른 이유

- 데이터가 적은 상황에 맞다. 처음 나온 DETR은 COCO로 500 epoch을 학습할 만큼 수렴이 느리고 데이터가 많이 필요하다. 우리 train은 185장뿐이다. RT-DETR는 COCO로 사전학습된 가중치(`rtdetr-l.pt`)에서 시작하는 실시간 DETR이라 적은 데이터로 미세조정하기에 낫다.
- 같은 조건으로 비교할 수 있다. RT-DETR는 YOLO26과 같은 Ultralytics 라이브러리로 학습한다. 그래서 같은 분할, 같은 YOLO 형식 데이터, 같은 평가 방법을 쓸 수 있고, 점수 차이를 모델 차이로 보기 쉽다.

다른 transformer 후보와 비교

| 후보 | 고르지 않은 이유 |
|---|---|
| RF-DETR | 최신 모델이지만 COCO 형식 데이터가 필요해서 변환을 따로 만들어야 하고, 문서상 입력 크기가 384~640이다. |
| D-FINE | 위치 정밀도를 높인 모델이지만, 학습 반복문과 평가를 직접 짜야 해서 베이스라인으로는 품이 많이 든다. |

#### 한계

- Ultralytics가 제공하는 사전학습 가중치는 큰 모델(L, X)뿐이라 YOLO26의 작은 모델보다 무겁고 느리다.
- 공식 문서에 따르면 GPU에서 RT-DETR는 `deterministic=False`로 학습해야 한다. 그래서 seed가 같아도 결과가 조금씩 달라질 수 있다. 여러 seed로 학습해서 평균으로 비교한다.
- 2023년에 나온 구조라, 이후에 나온 DETR 계열(RF-DETR, D-FINE 등)보다 성능이 낮을 수 있다.

## 2. 데이터 준비

Ultralytics는 YOLO 형식 데이터만 읽는다. 지금 데이터는 알약 하나가 JSON 하나인 COCO 형식이라, 아래 순서로 바꾼다. 방법은 팀의 YOLO26 베이스라인과 같아서, 두 모델이 같은 데이터로 학습한다.

| 단계 | 하는 일 |
|---|---|
| 2-1 | 원본 JSON을 읽어 bbox 표를 만들고, 클래스 번호표(category_id ↔ 0~55)를 만든다. |
| 2-2 | 라벨 수정 목록을 적용한다. |
| 2-3 | 분할 파일을 붙인다. |
| 2-4 | YOLO 형식(사진, txt 라벨, `data.yaml`)으로 바꾼다. |
| 2-5 | 바꾼 라벨을 사진에 다시 그려서 맞는지 본다. |

### 2-1. 라벨 읽기와 클래스 번호표

- `train_annotations`의 JSON을 모두 읽어 bbox 하나가 한 줄인 표를 만든다. bbox는 왼쪽 위 꼭짓점 기준 픽셀 `[x, y, w, h]`다.
- YOLO 형식은 클래스를 0부터 차례로 붙인 번호로 쓴다. 약 번호(category_id)를 작은 순으로 0, 1, 2, …에 대응시킨다. 제출할 때는 같은 표로 약 번호로 되돌린다.

In [ ]:
rows = []
cat_names = {}  # 약 번호 → 약 이름
for jp in sorted(TRAIN_ANN_DIR.rglob("*.json")):
    d = json.loads(jp.read_text(encoding="utf-8-sig"))  # utf-8-sig: 파일 앞에 BOM이 있어도 읽는다
    img = d["images"][0]
    for c in d["categories"]:
        cat_names[c["id"]] = c["name"]
    for a in d["annotations"]:
        rows.append({"file_name": img["file_name"], "img_w": img["width"], "img_h": img["height"],
                     "category_id": a["category_id"], "bbox": a["bbox"]})
ann = pd.DataFrame(rows)

CAT_IDS = sorted(cat_names)
CAT2YOLO = {c: i for i, c in enumerate(CAT_IDS)}  # 약 번호 → 0~55
YOLO2CAT = {i: c for c, i in CAT2YOLO.items()}    # 0~55 → 약 번호

print(f"bbox: {len(ann)}개 / 이미지: {ann['file_name'].nunique()}장 / 클래스: {len(CAT_IDS)}개")
print("번호표 앞부분:", list(CAT2YOLO.items())[:5])
print("같은 (사진, 클래스) bbox가 2개 이상:", int(ann.duplicated(["file_name", "category_id"]).sum()), "건")

### 2-2. 라벨 수정 목록 적용

- 팀 공용 라벨 수정 목록을 적용한다. 한 줄이 bbox 하나이고, 열은 `file_name, category_id, action, bbox, area`다. `bbox`와 `area`는 고친 뒤의 값이다.
- `add`: 빠진 bbox를 추가한다.
- `modify`: 같은 사진, 같은 클래스의 bbox를 찾아 목록의 bbox로 바꾼다.
    - 2-1에서 한 사진에 같은 클래스 bbox가 2개 이상인 경우가 없음을 확인했으므로, (사진, 클래스)로 bbox 하나를 찾을 수 있다.
    - 목록의 bbox가 지금 bbox와 같은 줄은 면적(area) 값만 고친 줄이다. 학습에는 bbox만 쓰고 area는 쓰지 않아서 건너뛴다.
- 목록에 고치기 전 bbox가 없어서, 목록이 같은 원본을 기준으로 만들어졌는지는 고칠 bbox를 모두 찾았는지와 적용 뒤 bbox 수로 확인한다. 적용 뒤 bbox는 771개(원본 763개 + 추가 8개)여야 한다.

In [ ]:
fixes = pd.read_csv(FIX_CSV, encoding="utf-8-sig")
assert {"file_name", "category_id", "action", "bbox"} <= set(fixes.columns), f"라벨 수정 목록의 열이 다릅니다: {list(fixes.columns)}"
img_size = ann.drop_duplicates("file_name").set_index("file_name")[["img_w", "img_h"]]

# 표의 칸에 리스트(bbox)를 직접 바꿔 넣지 않도록, 행 목록으로 바꿔서 고친 뒤 다시 표로 만든다
records = ann.to_dict("records")
row_of = {(r["file_name"], r["category_id"]): i for i, r in enumerate(records)}  # (사진, 클래스) → 행 번호

applied = Counter()
for f in fixes.itertuples():
    key = (f.file_name, int(f.category_id))
    bbox = json.loads(f.bbox)  # "[x, y, w, h]" 글자 → 리스트
    if f.action == "add":
        assert key not in row_of, f"이미 있는 bbox를 추가하려 합니다: {key}"
        w, h = img_size.loc[f.file_name]
        records.append({"file_name": f.file_name, "img_w": w, "img_h": h,
                        "category_id": int(f.category_id), "bbox": bbox})
        row_of[key] = len(records) - 1
        applied["add"] += 1
    elif f.action == "modify":
        assert key in row_of, f"고칠 bbox를 못 찾았습니다: {key}"
        r = records[row_of[key]]
        if r["bbox"] == bbox:
            applied["modify (bbox 같음, area만 수정 → 건너뜀)"] += 1
        else:
            r["bbox"] = bbox
            applied["modify (bbox 수정)"] += 1
    else:
        raise ValueError(f"알 수 없는 action: {f.action}")
ann = pd.DataFrame(records)

for k, v in applied.items():
    print(f"{k}: {v}줄")
print(f"목록 전체: {len(fixes)}줄")
print(f"bbox: {len(ann)}개 / 이미지: {ann['file_name'].nunique()}장")
assert sum(applied.values()) == len(fixes), "적용하지 않은 줄이 있습니다."
assert len(ann) == 771, f"bbox가 771개가 아닙니다: {len(ann)}"

### 2-3. 분할 붙이기

- `split.csv`는 사진마다 `train` 또는 `val`이 적힌 표다(열: `file_name`, `combo`, `split`).
- 다음을 확인한다.
    - 사진 232장이 모두 들어 있고, train 185장, val 47장인지
    - 같은 조합의 사진이 train과 val에 나뉘지 않았는지(조합 단위 분할)
    - 분할 정보가 빠진 bbox가 없는지

In [ ]:
split = pd.read_csv(SPLIT_CSV)
n_split = {k: int(v) for k, v in split["split"].value_counts().items()}
print("사진 수:", n_split)
combo_splits = split.groupby("combo")["split"].nunique()
print(f"train과 val에 나뉜 조합: {int((combo_splits > 1).sum())}개 / 조합 {len(combo_splits)}개")

ann = ann.merge(split[["file_name", "split"]], on="file_name", how="left")
print("분할 정보가 빠진 bbox:", int(ann["split"].isna().sum()))
display(ann.groupby("split").agg(사진=("file_name", "nunique"), bbox=("file_name", "size"), 클래스=("category_id", "nunique")))

assert len(split) == 232 and n_split == {"train": 185, "val": 47}, "분할이 1차 EDA(seed 410)와 다릅니다."
assert (combo_splits == 1).all(), "train과 val에 나뉜 조합이 있습니다."
assert ann["split"].notna().all(), "분할 정보가 빠진 bbox가 있습니다."

### 2-4. YOLO 형식으로 변환

- 사진 한 장에 txt 한 개를 만든다. 한 줄이 bbox 하나이고, `클래스번호 중심x 중심y 너비 높이`를 이미지 크기로 나눈 0~1 값으로 적는다.
- 사진은 `images/train`, `images/val`로 복사한다. Ultralytics는 사진 경로의 `images`를 `labels`로 바꿔 라벨을 찾기 때문이다.
- 이미지 밖으로 나간 bbox는 경계에서 자르고, 완전히 밖이면 버린다. 라벨 수정으로 이미지 밖 bbox(3-2의 18357)는 고쳐져서, 버리는 bbox는 0개여야 한다.
- `data.yaml`에는 폴더 경로와 클래스 이름을 적는다. 클래스 이름은 한글이 깨지지 않게 약 번호를 글자로 쓴다.
- 다시 실행할 때 예전 파일이 남지 않게 `data/processed/yolo`를 지우고 새로 만든다. 팀의 YOLO26 노트북과 같은 폴더, 같은 방법이라 만들어지는 내용도 같다.

In [ ]:
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for s in ["train", "val"]:
    (YOLO_DIR / "images" / s).mkdir(parents=True)
    (YOLO_DIR / "labels" / s).mkdir(parents=True)

dropped = 0
for name, g in ann.groupby("file_name"):
    s = g["split"].iloc[0]
    shutil.copy(TRAIN_IMG_DIR / name, YOLO_DIR / "images" / s / name)

    lines = []
    for r in g.itertuples():
        x, y, w, h = r.bbox
        W, H = r.img_w, r.img_h
        x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)  # 이미지 밖으로 나간 부분은 자른다
        if x2 <= x1 or y2 <= y1:
            dropped += 1  # 완전히 밖이면 버린다
            continue
        cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
        bw, bh = (x2 - x1) / W, (y2 - y1) / H
        lines.append(f"{CAT2YOLO[r.category_id]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    (YOLO_DIR / "labels" / s / (Path(name).stem + ".txt")).write_text("\n".join(lines))

data_cfg = {
    "path": str(YOLO_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {i: str(c) for i, c in YOLO2CAT.items()},
}
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

for s in ["train", "val"]:
    n_img = len(list((YOLO_DIR / "images" / s).glob("*.png")))
    n_lbl = len(list((YOLO_DIR / "labels" / s).glob("*.txt")))
    print(f"{s}: 사진 {n_img}장 / 라벨 {n_lbl}개")
saved = sum(len(p.read_text().splitlines()) for p in (YOLO_DIR / "labels").rglob("*.txt"))
print(f"저장한 bbox: {saved}개 / 버린 bbox: {dropped}개 / 표의 bbox: {len(ann)}개")
print("data.yaml:", rel(DATA_YAML))
assert saved + dropped == len(ann), "저장한 bbox 수가 표와 맞지 않습니다."

### 2-5. 변환 확인

- YOLO txt를 다시 읽어 픽셀 좌표로 되돌린 뒤 사진에 그린다. bbox가 알약을 감싸면 변환 공식이 맞다.
- train과 val에서 2장씩 무작위로 고른다. seed를 고정해서 다시 실행해도 같은 사진이 나온다.

In [ ]:
SAMPLE_SEED = 0
sample_rng = np.random.default_rng(SAMPLE_SEED)
samples = []
for s in ["train", "val"]:
    paths = sorted((YOLO_DIR / "images" / s).glob("*.png"))
    samples += [(s, paths[i]) for i in sorted(sample_rng.choice(len(paths), 2, replace=False))]

fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 5.5), layout="constrained")
for ax, (s, img_path) in zip(axes, samples):
    with Image.open(img_path) as im:
        W, H = im.size
        ax.imshow(im)
    for line in (YOLO_DIR / "labels" / s / (img_path.stem + ".txt")).read_text().splitlines():
        c, cx, cy, bw, bh = line.split()
        bw, bh = float(bw) * W, float(bh) * H                     # 0~1 비율 → 픽셀
        x1, y1 = float(cx) * W - bw / 2, float(cy) * H - bh / 2   # 중심 → 왼쪽 위
        ax.add_patch(Rectangle((x1, y1), bw, bh, fill=False, edgecolor="lime", linewidth=2))
        ax.text(x1, y1 - 10, YOLO2CAT[int(c)], color="white", fontsize=8,
                bbox=dict(facecolor="green", edgecolor="none", pad=1))
    ax.set_title(f"{s}: {img_path.stem.split('_')[0]}", fontsize=8)
    ax.axis("off")
fig.suptitle("YOLO 라벨을 다시 그린 사진 (초록 사각형: bbox, 숫자: 약 번호)", fontsize=11)
plt.show()

### 2번 결과

| 단계 | 결과 |
|---|---|
| 2-1 라벨 읽기 | bbox 763개, 사진 232장, 클래스 56개. 한 사진에 같은 클래스 bbox가 2개 이상인 경우는 없다. |
| 2-2 라벨 수정 | add 8줄로 bbox를 추가하고, modify 13줄 중 bbox가 바뀐 9줄을 고쳤다. bbox가 같은 4줄(area만 수정)은 건너뛰었다. bbox는 771개가 되었다. |
| 2-3 분할 | train 185장(bbox 612개, 클래스 56개), val 47장(bbox 159개, 클래스 30개). train과 val에 나뉜 조합, 분할 정보가 빠진 bbox는 없다. |
| 2-4 YOLO 변환 | txt 라벨 232개(train 185, val 47)에 bbox 771개를 저장했고, 버린 bbox는 없다. |
| 2-5 변환 확인 | train, val에서 2장씩 YOLO 라벨을 다시 그렸을 때 4장 모두 bbox가 알약을 감쌌다. 변환 공식이 맞다. |

- val에는 56종 중 30종만 있다. Ultralytics는 val 정답에 있는 클래스만 평균하므로, 뒤에 나오는 val 점수는 이 30종에 대한 점수다. train에만 있는 26종은 val로 평가되지 않는다.

## 3. 학습 설정

### 3-1. 학습 인자

- 팀의 YOLO26 본 학습(Colab)과 같은 값을 쓴다. 다른 것은 모델, 그리고 아래 표에 이유를 적은 `workers`, `deterministic`뿐이다.
- 표에 없는 인자(학습률, 증강 세기 등)는 Ultralytics 기본값을 쓴다. 학습이 끝나면 실제로 쓴 값 전체가 run 폴더의 `args.yaml`에 저장된다.

| 인자 | 값 | 뜻 | 이 값을 쓰는 이유 |
|---|---|---|---|
| 모델 | `rtdetr-l.pt` | COCO로 사전학습된 RT-DETR-L 가중치 | Ultralytics가 주는 RT-DETR 가중치(L, X) 중 작은 쪽 |
| epochs | 100 | train 사진 전체를 몇 번 반복해서 볼지 | 팀 설정 |
| patience | 30 | val 성능(mAP50-95)이 30 epoch 동안 나아지지 않으면 일찍 멈춘다 | 팀 설정 |
| imgsz | 640 | 사진을 640×640 크기로 맞춰 넣는다. RT-DETR는 가로세로 비율을 유지하지 않고 늘이거나 줄여서 맞춘다(학습, val, 예측 모두 같다) | 팀 설정. 원본 976×1280이 640×640이 된다(976 쪽 0.66배, 1280 쪽 0.5배). 1차 EDA 5-4는 비율을 유지해 둘 다 0.5배로 줄인 경우를 봤고 각인이 보였다. 여기서는 976 쪽이 그보다 덜 줄어든다 |
| batch | 16 | 한 번에 GPU에 올리는 사진 수 | 팀 설정. GPU 메모리가 모자라면 8로 줄인다 |
| optimizer | auto | 가중치를 고치는 방법과 학습률을 Ultralytics가 고른다 | 팀 설정. 아래에 설명 |
| 증강 | 기본값 | 학습 사진을 무작위로 바꿔(좌우 뒤집기, 색, 크기, 모자이크 등) 다양하게 만든다 | 팀 설정 |
| seed | 0, 1, 2 | 학습의 무작위 요소(새로 붙는 층의 초깃값, 사진 순서, 증강)를 정하는 값 | 팀 설정. seed마다 점수가 얼마나 흔들리는지 본다 |
| workers | 0 | 사진을 읽는 보조 프로세스 수 | Windows 주피터에서는 0이 아니면 오류가 날 수 있다. 속도에만 영향이 있다(팀원은 Colab에서 2) |
| deterministic | False | 같은 seed면 같은 결과가 나오도록 강제할지 | RT-DETR가 쓰는 연산(`grid_sample`)이 이 옵션을 지원하지 않는다 |

optimizer=auto가 고르는 값 (Ultralytics 8.4.183 기준)
- 전체 반복 수가 10,000 이하이면 AdamW를 고른다. 반복 수는 `올림(train 사진 수 / max(batch, 64)) × epochs` = 올림(185 / 64) × 100 = 300이라 AdamW가 된다.
- 학습률도 `0.002 × 5 / (4 + 클래스 수)` = 0.002 × 5 / 60 ≈ 0.000167로 정한다. 이때 `lr0`(기본 0.01)는 쓰지 않는다.
- 64는 `nbs`(기준 batch)다. batch 16이면 4번 모아서 가중치를 한 번 고친다(기울기 누적). batch를 8로 줄여도 8번 모아서 고치므로, 한 번에 반영되는 사진 수는 64장으로 같다.

AMP
- AMP(자동 혼합 정밀도)는 기본값대로 켠다. 일부 계산을 16비트로 해서 메모리를 덜 쓰고 빨라진다.
- Ultralytics RT-DETR 학습 코드 주석에 AMP에서 손실이 NaN이 될 수 있다고 적혀 있어서, 확인 실행에서 손실이 숫자로 나오는지 본다.

짧은 확인 실행
- `SMOKE_TEST = True`이면 seed 0으로 1 epoch만 돈다. 본 학습(seed당 최대 100 epoch) 전에 batch 16이 GPU 메모리에 들어가는지, W&B에 기록되는지 먼저 보기 위해서다.
- 확인 실행은 이름 끝에 `_smoke`를 붙여 본 학습 결과와 섞이지 않게 한다.
- 문제가 없으면 `False`로 바꾸고 이 셀부터 다시 실행한다.

In [ ]:
SMOKE_TEST = False  # True: seed 0으로 1 epoch만 도는 확인 실행 / False: 본 학습

MODEL_NAME = "rtdetr-l"
WEIGHTS = WEIGHTS_DIR / f"{MODEL_NAME}.pt"  # COCO 사전학습 가중치. 없으면 처음 학습할 때 내려받는다
EPOCHS = 100
PATIENCE = 30
IMGSZ = 640
BATCH = 16
SEEDS = [0, 1, 2]


def make_run_name(seed):
    """run 폴더 이름. 팀원 실행 이름(yolo26s_e100_s0_fix)과 형식을 맞춘다. _fix는 라벨 수정 반영이라는 뜻"""
    name = f"{MODEL_NAME}_e{EPOCHS}_s{seed}_fix"
    return f"{name}_smoke" if SMOKE_TEST else name


def train_args(seed):
    """model.train()에 넘길 인자. 여기 없는 인자는 Ultralytics 기본값을 쓴다."""
    return dict(
        data=str(DATA_YAML),
        epochs=1 if SMOKE_TEST else EPOCHS,
        patience=PATIENCE,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        seed=seed,
        workers=0,            # Windows 주피터에서 사진 읽기 오류를 피하려고 0
        deterministic=False,  # RT-DETR의 grid_sample이 deterministic을 지원하지 않는다
        project=str(RUNS_DIR),
        name=make_run_name(seed),
        exist_ok=True,        # 같은 이름의 run 폴더가 있으면 덮어쓴다
    )


print("모드:", "확인 실행 (seed 0, 1 epoch)" if SMOKE_TEST else f"본 학습 (seed {SEEDS}, 최대 {EPOCHS} epoch)")
print("사전학습 가중치:", rel(WEIGHTS), "(있음)" if WEIGHTS.exists() else "(없음, 처음 학습할 때 내려받음)")
print()
for k, v in train_args(0).items():
    if k not in ("data", "project"):  # 경로는 0번에서 출력했다
        print(f"{k:14}: {v}")

### 3-2. W&B 기록

- 팀 W&B(`codeit-prj1-team2` / `pill-object-detection`)에 seed마다 실행 하나로 기록한다.
    - 실행 이름: `jw_` + run 폴더 이름. 예: `jw_rtdetr-l_e100_s0_fix`
    - 태그: `rtdetr`, `label-fixed`, `baseline`(확인 실행은 `smoke`)
    - config: 모델, 분할 seed(410), 데이터 설명, 3-1의 학습 인자
- Ultralytics가 자동으로 기록하는 것: epoch마다 손실, 학습률, `metrics/mAP50(B)`, `metrics/mAP50-95(B)` 등. 팀원 YOLO26 실행과 같은 이름이라 바로 비교할 수 있다. 끝에 그래프와 best.pt도 올린다.
- 대회 지표(mAP75-95)는 자동 기록에 없어서 콜백으로 넣는다. 콜백은 학습 중 정해진 시점에 Ultralytics가 불러 주는 함수다.
    - `on_fit_epoch_end`(val이 끝날 때마다): IoU 0.75~0.95 다섯 단계 AP의 평균을 `val/map_75_95`로 기록한다.
    - `on_train_end`(학습이 끝날 때): best.pt로 다시 잰 값을 요약값 `best_val/map_75_95`로 넣는다.
    - 두 이름은 팀원 Faster R-CNN 실행과 같다. 다만 여기 best.pt는 mAP50-95로 고른 것이고, Faster R-CNN 값은 mAP75-95가 가장 높은 epoch의 값이다. 계산 라이브러리도 다르다(Ultralytics, torchmetrics). 그래서 모델끼리는 Kaggle 점수로 비교한다.
- W&B 실행은 학습 전에 직접 시작한다(`wandb.init`). Ultralytics에 맡기면 `project`에 넣은 결과 폴더 경로가 W&B 프로젝트 이름이 되어, 사람마다 다른 프로젝트에 흩어지고 내 컴퓨터 경로도 W&B에 남는다.
- W&B 로그인은 터미널에서 `wandb login`으로 한 번 해 둔다. 노트북에는 키를 적지 않는다.

In [ ]:
WANDB_ENTITY = "codeit-prj1-team2"      # W&B 팀
WANDB_PROJECT = "pill-object-detection"
WANDB_PREFIX = "jw"                      # 실행 이름 앞에 붙이는 이니셜 (팀원: cy_)
SPLIT_SEED = 410                         # split.csv를 만든 분할 seed (기록용)


def map75_95(metrics):
    """대회 지표 근사: IoU 0.75, 0.80, 0.85, 0.90, 0.95에서 AP를 구해 평균한다.
    metrics.box.all_ap는 (val 정답에 있는 클래스 수, IoU 0.50~0.95 10단계) 표라 뒤 5칸만 쓴다."""
    ap = metrics.box.all_ap
    return float(ap[:, 5:].mean()) if len(ap) else float("nan")


def log_map75_95(trainer):
    """콜백: val이 끝날 때마다 불린다. Ultralytics의 W&B 기록과 같은 단계(epoch + 1)에 넣는다."""
    if wandb.run is not None:
        wandb.run.log({"val/map_75_95": map75_95(trainer.validator.metrics)}, step=trainer.epoch + 1)


def save_best_map75_95(trainer):
    """콜백: 학습이 끝날 때 불린다. 이때 validator에는 best.pt로 다시 잰 결과가 들어 있다."""
    if wandb.run is not None:
        wandb.run.summary["best_val/map_75_95"] = map75_95(trainer.validator.metrics)


def wandb_config(args):
    """W&B config. 팀원 YOLO26 실행과 같은 구조로 넣는다. 경로는 넣지 않는다."""
    return {
        "model": MODEL_NAME,
        "split_seed": SPLIT_SEED,
        "data": "라벨 수정 반영(bbox 771) + split.csv(seed 410)",
        **{k: v for k, v in args.items() if k not in ("data", "project", "name")},
    }


print("W&B:", f"{WANDB_ENTITY}/{WANDB_PROJECT}", "/ 실행 이름 예:", f"{WANDB_PREFIX}_{make_run_name(0)}")

## 4. 학습

### 4-1. GPU 메모리 확인

- 학습 전에 GPU에 남은 메모리를 본다. 다른 프로그램(브라우저, 게임 등)이 GPU 메모리를 쓰고 있으면 학습에 쓸 수 있는 양이 줄어든다.
- 학습 중 val은 batch의 2배(32장)로 돈다.
- Windows에서는 GPU 메모리가 모자라도 오류 없이 시스템 메모리를 빌려 쓰면서 학습이 크게 느려질 수 있다. 그래서 확인 실행에서 epoch 시간도 같이 본다.

In [ ]:
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU 메모리: 남은 {free / 1024**3:.1f} GiB / 전체 {total / 1024**3:.1f} GiB")
else:
    print("GPU가 없습니다.")

### 4-2. 학습 함수

seed 하나를 학습하는 함수 `train_one`을 만든다. 순서는 다음과 같다.

1. 사전학습 가중치로 모델을 만들고, 3-2의 콜백 두 개를 등록한다.
2. Ultralytics의 W&B 기록을 켜고, W&B 실행을 직접 시작한다.
3. 학습한다. 학습이 끝나면 Ultralytics가 best.pt로 val을 한 번 더 하고 W&B 실행을 닫는다.
4. Ultralytics의 W&B 설정을 원래 값으로 돌린다. 이 설정은 내 컴퓨터의 Ultralytics 설정 파일에 저장되어 다른 노트북에도 남기 때문이다. 학습 전에 오류가 나서 W&B 실행이 열려 있으면 여기서 닫는다.

끝나면 best.pt의 val 지표(mAP50, mAP50-95, mAP75-95)를 출력한다. 자세한 분석은 6번에서 한다.

In [ ]:
def train_one(seed):
    args = train_args(seed)

    model = RTDETR(str(WEIGHTS))
    model.add_callback("on_fit_epoch_end", log_map75_95)
    model.add_callback("on_train_end", save_best_map75_95)

    wandb_before = settings["wandb"]
    settings.update({"wandb": True})
    wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=f"{WANDB_PREFIX}_{args['name']}",
        tags=["rtdetr", "label-fixed", "smoke" if SMOKE_TEST else "baseline"],
        config=wandb_config(args),
    )
    try:
        metrics = model.train(**args)  # best.pt로 다시 잰 val 결과가 돌아온다
    finally:
        settings.update({"wandb": wandb_before})
        if wandb.run is not None:
            wandb.finish()

    print(f"\n[{args['name']}] best.pt val: mAP50 {metrics.box.map50:.4f} / "
          f"mAP50-95 {metrics.box.map:.4f} / mAP75-95 {map75_95(metrics):.4f}")
    return metrics

### 4-3. seed 0 학습

- 확인 실행(`SMOKE_TEST = True`)도 이 셀로 한다.
- 확인 실행에서 볼 것
    - 오류 없이 끝나는지. `CUDA out of memory`가 나면 3-1에서 batch를 8로 줄인다.
    - 출력 앞부분에 `optimizer: AdamW(lr=0.000167, ...)`가 나오는지
    - 진행 표시줄의 손실 값이 `nan`이 아닌지
    - 진행 표시줄의 `GPU_mem`(PyTorch가 잡은 GPU 메모리)과 epoch 시간. `GPU_mem`이 12G에 가깝거나 넘으면 시스템 메모리를 빌려 쓰고 있을 수 있다.
    - 출력에 나온 W&B 링크에서 팀 프로젝트에 실행이 생기고, `val/map_75_95`와 요약값 `best_val/map_75_95`가 들어갔는지
- 1 epoch로는 거의 학습되지 않으므로 확인 실행의 점수는 보지 않는다.

In [ ]:
metrics_s0 = train_one(0)

### 4-4. seed 1 학습

seed 0 본 학습이 끝난 뒤 실행한다. 확인 실행에서는 돌리지 않는다.

In [ ]:
assert not SMOKE_TEST, "확인 실행은 seed 0만 돈다. 3-1에서 SMOKE_TEST = False로 바꾼 뒤 실행하세요."
metrics_s1 = train_one(1)

### 4-5. seed 2 학습

seed 1 학습이 끝난 뒤 실행한다.

In [ ]:
assert not SMOKE_TEST, "확인 실행은 seed 0만 돈다. 3-1에서 SMOKE_TEST = False로 바꾼 뒤 실행하세요."
metrics_s2 = train_one(2)

### 4번 결과

- 세 seed 모두 100 epoch을 끝까지 돌았다. patience 30에 걸려 일찍 멈춘 seed는 없다.
- 3-1에서 계산한 대로 optimizer는 AdamW(lr=0.000167)로 정해졌다. AMP 확인을 통과했고, 손실이 NaN으로 나온 epoch은 없다.
- 사전학습 가중치에서 941개 중 926개를 가져왔다. 못 가져온 15개는 클래스 수(COCO 80종 → 56종)에 따라 크기가 달라지는 층(클래스 점수를 내는 층 등)이라 새로 학습한다.

| seed | mAP50 | mAP50-95 | mAP75-95 | 학습 시간 |
|---|---|---|---|---|
| 0 | 0.995 | 0.987 | 0.980 | 1.07시간 |
| 1 | 0.993 | 0.985 | 0.978 | 1.55시간 |
| 2 | 0.980 | 0.976 | 0.973 | 2.76시간 |
| 평균 | 0.989 | 0.983 | 0.977 | |

- 점수는 학습이 끝난 뒤 best.pt(mAP50-95가 가장 높은 epoch의 가중치)로 val 47장을 다시 잰 값이다.
- seed에 따른 mAP75-95 차이는 0.01보다 작다(0.973~0.980).
- 학습 시간은 seed마다 크게 달랐다. 학습 중 `GPU_mem`이 12.8G로 전용 GPU 메모리(12GB)를 넘어 시스템 메모리를 같이 썼고, seed 2는 중간에 batch 하나에 10초 넘게 걸린 구간이 있었다. 학습 시간은 점수에 영향을 주지 않는다.

클래스별 결과 (학습 끝에 Ultralytics가 출력한 표, val에 있는 30종)

- Ultralytics 계산 방식에서는 AP의 최고값이 0.995다. 30종 대부분은 세 seed 모두 mAP50과 mAP50-95가 0.995다.
- 세 seed 모두 mAP50-95가 0.99보다 낮은 클래스는 16262(0.963 / 0.963 / 0.945), 18147(0.901 / 0.955 / 0.967), 25469(0.983 / 0.937 / 0.978)다. 셋 다 mAP50은 0.995라서, 약을 찾고 클래스도 맞혔지만 IoU 기준을 높이면 박스가 덜 맞는 것으로 보인다.
- 30308은 seed마다 크게 다르다(mAP50-95 0.995 / 0.924 / 0.537). seed 2는 mAP50도 0.537이라, 박스 위치보다 약을 놓쳤거나 다른 약으로 예측한 것으로 보인다.
- recall이 낮은데 mAP는 높은 클래스가 있다. 20238은 recall이 0.23 / 0.66 / 0.38인데 mAP50-95는 세 seed 모두 0.995다. Ultralytics의 precision과 recall은 전체 F1이 가장 높은 신뢰도 기준값 하나에서 잰 값이라, 이 약은 그 기준값보다 낮은 점수로 맞혔다는 뜻이다. 그래서 제출 CSV에는 점수가 낮은 예측도 넣어야 한다.
- precision이 낮은 클래스는 22347(0.41 / 0.74 / 0.50)이다. 22347로 예측한 박스 중 정답이 아닌 것이 많다는 뜻으로, 다른 약을 22347로 봤거나 같은 약에 박스를 겹쳐 냈을 수 있다.
- 위 해석은 숫자만 보고 한 것이다. 6번에서 val 예측을 다시 해서 박스 단위로 확인한다.

W&B 기록

- seed 1, 2를 학습하는 동안 W&B 서버 오류(Service Unavailable, 500)가 여러 번 났다. 학습은 영향 없이 끝났고 마지막에 동기화도 끝났다고 출력됐지만, W&B 화면의 seed 1 요약값은 89 epoch에서 멈춰 있었다. 결과는 로컬 run 폴더(`runs/rtdetr-l_e100_s*_fix`)를 기준으로 본다.

## 5. test 추론과 제출

- 세 seed의 best.pt로 test 사진 전체를 예측하고, 대회 형식 CSV로 저장해 Kaggle에 각각 제출한다.
- test에는 정답이 없다. 그래서 이 섹션에서는 제출 파일의 형식과 예측의 모양(사진당 박스 수)만 보고, 성능은 Kaggle 점수로 본다.
- 학습은 다시 하지 않는다. 커널을 다시 시작했다면 0번 두 셀과 3-1, 3-2를 실행한 뒤 이 섹션을 실행한다.

### 5-1. test 예측과 CSV 저장

- 입력 크기는 학습과 같은 640이다. 예측할 때도 사진을 640×640으로 늘려 맞추고, 예측한 박스는 Ultralytics가 원본 크기로 되돌린다.
- 신뢰도 0.01 이상인 예측을 모두 넣는다. AP는 클래스마다 예측을 점수 순으로 줄 세워 계산하므로, 정답보다 점수가 낮은 오답은 점수를 깎지 않고, 점수가 낮은 정답은 점수를 올린다. 4번 결과의 20238처럼 낮은 점수로 맞힌 약이 있었다.
- 클래스 번호(0~55)는 모델에 저장된 클래스 이름으로 약 번호로 되돌린다. 2-4에서 `data.yaml`의 클래스 이름을 약 번호로 적었기 때문이다.
- 박스는 꼭짓점 좌표를 사진 안으로 자른 뒤 `[x, y, w, h]` 픽셀로 바꾼다. 정답 박스는 사진 안에 있으므로, 사진 밖으로 나간 부분을 잘라도 겹치는 넓이는 그대로라 IoU가 줄지 않는다.
- `image_id`는 test 파일 이름의 숫자이고, `annotation_id`는 1부터 차례로 붙인다.
- 저장 위치는 `runs/<run 이름>/submission_<run 이름>.csv`다. 팀 제출 파일과 같은 이름 형식이다.

In [ ]:
SUB_COLS = ["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"]
PRED_CONF = 0.01  # 이 점수 이상인 예측을 모두 제출한다

test_paths = sorted(TEST_IMG_DIR.glob("*.png"))
assert all(p.stem.isdigit() for p in test_paths), "파일 이름이 숫자가 아닌 test 사진이 있습니다."
TEST_IDS = sorted(int(p.stem) for p in test_paths)
print("test 사진:", len(TEST_IDS), "장")


def submission_path(seed):
    run_name = make_run_name(seed)
    return RUNS_DIR / run_name / f"submission_{run_name}.csv"


def predict_test(seed):
    """seed의 best.pt로 test 사진을 예측해 제출 형식 표로 만든다."""
    model = RTDETR(str(RUNS_DIR / make_run_name(seed) / "weights" / "best.pt"))
    rows = []
    for r in model.predict(source=str(TEST_IMG_DIR), imgsz=IMGSZ, conf=PRED_CONF, device=DEVICE,
                           stream=True, verbose=False):  # stream: 사진을 한 장씩 처리해 메모리를 아낀다
        H, W = r.orig_shape
        image_id = int(Path(r.path).stem)
        for (x1, y1, x2, y2), c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            x1, y1, x2, y2 = max(0.0, x1), max(0.0, y1), min(W, x2), min(H, y2)  # 사진 밖으로 나간 부분은 자른다
            if x2 <= x1 or y2 <= y1:
                continue
            rows.append([image_id, int(model.names[int(c)]),  # 0~55 → 약 번호
                         round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2), round(s, 4)])
    sub = pd.DataFrame(rows, columns=SUB_COLS[1:])
    sub.insert(0, "annotation_id", range(1, len(sub) + 1))
    return sub


subs = {}
for seed in SEEDS:
    subs[seed] = predict_test(seed)
    subs[seed].to_csv(submission_path(seed), index=False)
    print(f"seed {seed}: {len(subs[seed])}행 → {rel(submission_path(seed))}")

### 5-2. 제출 파일 점검

- 형식: 열 순서, `annotation_id` 중복, 빈 값, 박스 너비와 높이가 0보다 큰지, score가 0~1인지, 약 번호가 학습한 56종 안에 있는지 확인한다. 하나라도 틀리면 멈춘다.
- 예측의 모양
    - 예측이 하나도 없는 test 사진 수
    - 사진당 박스 수(신뢰도 0.01 이상 전체)
    - 사진당 점수 0.5 이상인 박스 수. 0.5는 "모델이 자신 있게 낸 예측"을 세려고 정한 기준이다. 대회 사진에는 알약이 최대 4개라, 이 수가 알약 수보다 적으면 못 찾은 알약이 있고, 많으면 같은 알약에 겹친 예측이나 잘못 본 예측이 있다는 신호다.

In [ ]:
names = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))["names"]
TRAIN_CATS = {int(v) for v in names.values()}  # 학습한 56종의 약 번호

summary, conf_dist = [], []
for seed, sub in subs.items():
    assert list(sub.columns) == SUB_COLS, "열 순서가 다릅니다."
    assert sub["annotation_id"].is_unique, "annotation_id가 겹칩니다."
    assert sub.notna().all().all(), "빈 값이 있습니다."
    assert (sub[["bbox_w", "bbox_h"]] > 0).all().all(), "너비나 높이가 0 이하인 박스가 있습니다."
    assert sub["score"].between(0, 1).all(), "score가 0~1 밖에 있습니다."
    assert sub["category_id"].isin(TRAIN_CATS).all(), "학습하지 않은 약 번호가 있습니다."

    n_all = sub.groupby("image_id").size().reindex(TEST_IDS, fill_value=0)                         # 사진당 박스 수
    n_conf = sub[sub["score"] >= 0.5].groupby("image_id").size().reindex(TEST_IDS, fill_value=0)  # 사진당 0.5 이상 박스 수
    summary.append({"seed": seed, "행 수": len(sub), "예측 없는 사진": int((n_all == 0).sum()),
                    "사진당 박스(중앙값)": float(n_all.median()), "사진당 박스(최대)": int(n_all.max())})
    counts = n_conf.clip(upper=5).value_counts()
    conf_dist.append({"seed": seed, **{(f"{k}개" if k < 5 else "5개 이상"): int(counts.get(k, 0)) for k in range(6)}})

print("형식 점검 통과")
display(pd.DataFrame(summary).set_index("seed"))
print(f"사진당 점수 0.5 이상 박스 수별 test 사진 수 (전체 {len(TEST_IDS)}장)")
display(pd.DataFrame(conf_dist).set_index("seed"))

### 5-3. Kaggle 제출과 점수

- 노트북에서 Kaggle 명령줄 도구(`kaggle`)로 세 seed의 CSV를 각각 제출한다.
- 처음 한 번 준비할 것
    - `kaggle` 라이브러리 설치
    - Kaggle 계정 설정의 API 항목에서 토큰을 만들어 내 컴퓨터의 `.kaggle` 폴더에 저장한다. 토큰은 노트북에 적지 않는다.
    - 대회 규칙에 동의되어 있어야 제출할 수 있다.
- 제출 하나가 Kaggle 하루 제출 횟수 하나를 쓴다. `SUBMIT_SEEDS`에 적은 seed만 제출하고, 제출이 끝나면 `[]`로 바꿔 둔다. 그러면 셀을 다시 실행해도 제출하지 않는다.
- 제출 설명에는 run 이름을 넣어 Kaggle 제출 목록에서 구분한다.

In [ ]:
KAGGLE_COMP = "ai15-level1-project"  # 대회 이름 (팀원 노트북과 같다)
KAGGLE_CLI = Path(sys.executable).with_name("kaggle.exe" if platform.system() == "Windows" else "kaggle")  # 가상환경 안의 kaggle 명령


def kaggle_cli(*args):
    """kaggle 명령을 실행하고 출력을 셀에 보여 준다. 실패하면 오류 내용을 띄운다."""
    assert KAGGLE_CLI.exists(), "kaggle이 설치되어 있지 않습니다. 5-3 설명을 보고 설치하세요."
    r = subprocess.run([str(KAGGLE_CLI), *args], capture_output=True, text=True, encoding="utf-8",
                       errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    print(r.stdout.strip())
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip())


SUBMIT_SEEDS = [0, 1, 2]  # 제출할 seed. 제출이 끝나면 []로 바꿔 둔다
for seed in SUBMIT_SEEDS:
    print(f"--- seed {seed} 제출 ---")
    kaggle_cli("competitions", "submit", "-c", KAGGLE_COMP, "-f", str(submission_path(seed)),
               "-m", f"{make_run_name(seed)} RT-DETR-L baseline")

- 채점에는 1~2분쯤 걸린다. 아래 셀로 제출 목록과 점수를 본다. 점수가 아직 비어 있으면 잠시 뒤 다시 실행한다.

In [ ]:
kaggle_cli("competitions", "submissions", "-c", KAGGLE_COMP)